###Borrar y crear widgets

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "bakehouse_dev")

catalogName = dbutils.widgets.get("catalog")

# Bandera para mostrar prints en la fase de desarrollo
showPrint = False

### Insertar Silver - Customers

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

# Consultar tabla
df_bronze = spark.table(f"{catalogName}.bronze.customers_bronze")

# add metadata columns
df_bronze = df_bronze.withColumn("updatedate", F.current_timestamp())

if showPrint:
  row_count, column_count = df_bronze.count(), len(df_bronze.columns)
  print(f"Row count: {row_count}")
  print(f"Column count: {column_count}")
  display(df_bronze.limit(50))

Row count: 900
Column count: 13


customerID,first_name,last_name,email_address,phone_number,address,city,state,country,continent,postal_zip_code,gender,updatedate
2000259,Kayla,Barrett,brittanyramos@example.org,349-683-9514x73065,717 Whitney Roads,Kathrynborough,Massachusetts,Japan,Asia,81587,female,2026-02-16T18:24:14.757Z
2000260,Amanda,Reed,scollier@example.org,+1-999-308-9110,69075 Logan Circles Apt. 540,East Catherine,Rhode Island,Japan,Asia,6657,female,2026-02-16T18:24:14.757Z
2000261,Steven,Tanner,haileysanchez@example.net,859-946-4140x24086,08560 Thomas Land,Williamshire,Missouri,Japan,Asia,20642,female,2026-02-16T18:24:14.757Z
2000262,Jennifer,Forbes,belldonna@example.com,633-427-4977,5840 Warren Garden Suite 901,Delacruzville,Nevada,Australia,Oceania,21440,male,2026-02-16T18:24:14.757Z
2000263,Kenneth,Berger,bdalton@example.net,(831)220-1833x906,693 Baker Dale,West Wendy,Colorado,Australia,Oceania,32756,female,2026-02-16T18:24:14.757Z
2000264,James,Edwards,mary39@example.org,271-321-1561x9697,665 Campbell Streets Suite 966,Sherryton,Illinois,Japan,Asia,76717,male,2026-02-16T18:24:14.757Z
2000265,Ann,Montgomery,michaelreese@example.org,7673757334,220 Barber Islands Apt. 664,East Todd,Iowa,Australia,Oceania,77976,female,2026-02-16T18:24:14.757Z
2000266,Heather,Moore,mannmartin@example.com,278-606-3676x938,896 Greene Hill Suite 575,Josephburgh,Maryland,Australia,Oceania,2149,male,2026-02-16T18:24:14.757Z
2000267,Joseph,Graham,suzannereeves@example.org,538.787.9102x4963,2563 Jessica Mountains Apt. 192,Kelseyview,Mississippi,Australia,Oceania,85290,female,2026-02-16T18:24:14.757Z
2000268,Lisa,Ramirez,kristi04@example.com,(643)361-8721x12062,573 Bailey Lights Suite 941,West Deniseland,Connecticut,Japan,Asia,58198,female,2026-02-16T18:24:14.757Z


In [0]:
# Manejar Null en customerID
if showPrint:
    display(df_bronze.filter(F.col("customerID").isNull()).count())
    df_bronze.filter(F.col("customerID").isNull()).show(3)

# Drop rows where 'customerID' is null
df_bronze = df_bronze.dropna(subset=["customerID"])

# Get row count
if showPrint:
    row_count = df_bronze.count()
    print(f"Row count after droping null values: {row_count}")

0

+----------+----------+---------+-------------+------------+-------+----+-----+-------+---------+---------------+------+----------+
|customerID|first_name|last_name|email_address|phone_number|address|city|state|country|continent|postal_zip_code|gender|updatedate|
+----------+----------+---------+-------------+------------+-------+----+-----+-------+---------+---------------+------+----------+
+----------+----------+---------+-------------+------------+-------+----+-----+-------+---------+---------------+------+----------+

Row count after droping null values: 300


In [0]:
# Codigos repetidos
df_tmp = df_bronze.groupBy("customerID").agg(F.count("*").alias("count")).filter(F.col("count") > 1)
if showPrint:
  print(f"Filas totales: {df_bronze.count()}")
  print(f"Filas con duplicadas: {df_tmp.count()}")

# Remover filas repetidos

# Remueve filas sin tener en cuenta un orden especifico
# df_tmp = df_bronze.dropDuplicates(["customerID"])

# Apply the window function to add a row number column
window_spec = Window.partitionBy(["customerID"]).orderBy(F.col("email_address"), F.col("last_name"), F.col("first_name").desc())
df_tmp = df_bronze.withColumn("row_number", F.row_number().over(window_spec))

# Filter to keep only the first row within each partition (the one with rank 1)
df_tmp = df_tmp.filter(F.col("row_number") == 1).drop("row_number")
df_tmp = df_tmp.drop("row_number")
df_bronze = df_tmp
if showPrint:
  print(f"Filas despues de remover duplicados: {df_bronze.count()}")
  display(df_bronze.orderBy("customerID").limit(10))

Filas totales: 900
Filas con duplicadas: 300
Filas despues de remover duplicados: 300


customerID,first_name,last_name,email_address,phone_number,address,city,state,country,continent,postal_zip_code,gender,updatedate
2000000,Jimmy,Cross,ejacobson@example.net,+1-219-830-7162x208,0203 Amber Cliffs Apt. 330,Huntburgh,AS,USA,North America,85109,female,2026-02-16T18:24:17.289Z
2000001,Beth,Park,juliaflores@example.com,+1-928-749-9901,307 White Roads,Jacksonhaven,Oregon,Japan,Asia,74208,female,2026-02-16T18:24:17.289Z
2000002,Amber,Kirk,william99@example.net,4516676624,4281 Teresa Island,Scottland,TX,USA,North America,30555,female,2026-02-16T18:24:17.289Z
2000003,Charles,Parsons,hillkyle@example.com,(936)717-1247x77165,102 Wilson Canyon Apt. 344,Anthonyfurt,Hawaii,Australia,Oceania,94248,male,2026-02-16T18:24:17.289Z
2000004,Charles,Gonzalez,jessica67@example.net,928-258-5958x9952,6844 Richard Fords,East Shannonland,PW,USA,North America,45527,male,2026-02-16T18:24:17.289Z
2000005,Jared,Johnson,amy67@example.net,(690)957-6260x289,13910 Ray Mountains Suite 603,New Willieland,Kansas,Australia,Oceania,17081,female,2026-02-16T18:24:17.289Z
2000006,Christopher,Miller,johnyates@example.net,(722)867-8863x37789,20110 Dalton Square Suite 177,Tonyaville,West Virginia,Japan,Asia,24296,female,2026-02-16T18:24:17.289Z
2000007,Christopher,Garcia,zhangjohn@example.com,266.928.1866x8854,825 Erika Shoal Suite 862,Bakerside,Washington,Australia,Oceania,16133,male,2026-02-16T18:24:17.289Z
2000008,Omar,Zamora,amandacooper@example.net,001-592-508-2320x46021,18122 Sandra Valley Apt. 827,East Julian,NJ,USA,North America,54414,female,2026-02-16T18:24:17.289Z
2000009,Jacob,Lane,richard55@example.net,627-978-9352x59042,461 Sean Keys,Lake Michael,Hawaii,Australia,Oceania,48119,female,2026-02-16T18:24:17.289Z


In [0]:
# Estandarizar phone_number a formato válido de 14 caracteres: (ejemplo: (XXX)-XXX-XXX-XXXXX)
def standardize_phone(phone):
    if phone is None:
        return None
    # Eliminar caracteres no numéricos
    digits = re.sub(r'\D', '', phone)
    # Solo tomar los primeros 14 dígitos
    digits = digits[:14]
    return f"({digits[:3]})-{digits[3:6]}-{digits[6:9]}-{digits[9:]}"

standardize_phone_udf = F.udf(standardize_phone, StringType())

df_bronze = df_bronze.withColumn("phone_number", standardize_phone_udf(F.col("phone_number")))
if showPrint:
    display(df_bronze.select("phone_number").limit(10))

phone_number
(121)-983-071-62208
(192)-874-999-01
(451)-667-662-4
(936)-717-124-77716
(928)-258-595-89952
(690)-957-626-0289
(722)-867-886-33778
(266)-928-186-68854
(001)-592-508-23204
(627)-978-935-25904


In [0]:
# en el dataframe df_bronze cambiar la columna gender para que muestre un boleano, cuando tenga el texto female usar False y cuando tenga el texto male usar True. Cambiar el tipo de datos de la columna gender por boolean.
df_bronze = df_bronze.withColumn(
    "gender",
    F.when(F.col("gender") == "male", True)
    .when(F.col("gender") == "female", False)
    .otherwise(None)
)

df_tmp = df_bronze.filter(F.col("gender").isNull())
if showPrint:
    print(f"Cantidad de filas con gender = None: {df_tmp.count()}")
    display(df_bronze.select("customerID", "gender").limit(10))

Cantidad de filas con gender = None: 0


customerID,gender
2000000,false
2000001,false
2000002,false
2000003,true
2000004,true
2000005,false
2000006,false
2000007,true
2000008,false
2000009,false


In [0]:
# Convertir a mayusculas
df_bronze = df_bronze.withColumn("continent", F.upper(F.col("continent")))

if showPrint:
    display(df_bronze.select("customerID", "continent").limit(10))

customerID,continent
2000000,NORTH AMERICA
2000001,ASIA
2000002,NORTH AMERICA
2000003,OCEANIA
2000004,NORTH AMERICA
2000005,OCEANIA
2000006,ASIA
2000007,OCEANIA
2000008,NORTH AMERICA
2000009,OCEANIA


In [0]:
# en el dataframe df_bronze en la columna state remplazar codigos de estado de dos letras con los nombres de estados de USA
us_state_map = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia",
    "HI": "Hawaii", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa",
    "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi", "MO": "Missouri",
    "MT": "Montana", "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NY": "New York", "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming"
}

df_bronze = df_bronze.withColumn(
    "state",
    F.when(
        F.length(F.col("state")) == 2,
        F.create_map([F.lit(x) for pair in us_state_map.items() for x in pair])[F.col("state")]
    ).otherwise(F.col("state"))
)

if showPrint:
    display(df_bronze.select("state").distinct().orderBy("state"))

state
null
Alabama
Alaska
Arizona
Arkansas
California
Colorado
Connecticut
Delaware
Florida


In [0]:
# Seleccionar y organizar columnas
df_silver = df_bronze.select("customerID", "first_name", "last_name", "gender", "email_address", "phone_number", "address", "continent", "country", "state", "city", "postal_zip_code", "updatedate")

if showPrint:
    df_silver.printSchema()

root
 |-- customerID: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: boolean (nullable = true)
 |-- email_address: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- address: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postal_zip_code: long (nullable = true)
 |-- updatedate: timestamp (nullable = false)



In [0]:
# guardar los datos del dataframe en la tabla
df_silver.write.format("delta") \
    .mode("overwrite") \
    .options(mergeSchema="true") \
    .saveAsTable(f"{catalogName}.silver.customers_silver")